# 🔬 DermaFusion-AI — Kaggle Training Notebook
## EVA-02 Large + ConvNeXt V2 | Dual-Branch Fusion | SOTA 2026
### ⚙️ Before Running:
1. Datasets already added ✅
2. **Accelerator → GPU T4 x2** ✅
3. **Persistence → Files only** ✅
4. Run cells top to bottom — Steps 5+6 are combined to auto-start classifier after segmentation

## Step 0: Inspect Input Paths

In [ ]:
import os

def explore(path, depth=0, max_depth=2):
    if depth > max_depth or not os.path.exists(path):
        return
    for item in sorted(os.listdir(path)):
        full = os.path.join(path, item)
        kind = '📁' if os.path.isdir(full) else '📄'
        print('  ' * depth + f'{kind} {item}')
        if os.path.isdir(full):
            explore(full, depth + 1, max_depth)

explore('/kaggle/input', max_depth=2)

## Step 1: Clone / Pull Repo

In [ ]:
import os
if not os.path.exists('/kaggle/working/DermaFusion-AI'):
    !git clone https://github.com/ai-with-abdullah/DermaFusion-AI.git /kaggle/working/DermaFusion-AI
else:
    !cd /kaggle/working/DermaFusion-AI && git fetch origin && git reset --hard origin/main

%cd /kaggle/working/DermaFusion-AI
print('Working directory:', os.getcwd())


## Step 2: Install Dependencies

In [ ]:
%pip install -q timm>=0.9.12 albumentations>=1.3.1 opencv-python-headless scikit-learn scipy tqdm
print('✅ Dependencies installed')

## Step 3: Verify GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'✅ GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')
else:
    print('❌ No GPU — Settings > Accelerator > GPU T4 x2')

## Step 4: Link All Datasets → data/ folder

In [ ]:
import os

data_dir = '/kaggle/working/DermaFusion-AI/data'
os.makedirs(data_dir, exist_ok=True)

search_roots = ['/kaggle/input/competitions']
datasets_dir = '/kaggle/input/datasets'
if os.path.exists(datasets_dir):
    for username in os.listdir(datasets_dir):
        user_path = os.path.join(datasets_dir, username)
        if os.path.isdir(user_path):
            search_roots.append(user_path)
search_roots.append('/kaggle/input')

def find_folder(keywords):
    # Search up to 3 levels deep in /kaggle/input
    paths_to_check = ['/kaggle/input']
    checked_paths = set()
    for depth in range(3):
        next_paths = []
        for p in paths_to_check:
            if not os.path.exists(p) or p in checked_paths:
                continue
            checked_paths.add(p)
            try:
                for item in os.listdir(p):
                    full = os.path.join(p, item)
                    if os.path.isdir(full):
                        if any(kw in item.lower() for kw in keywords):
                            # Check for a deeper subfolder that matches
                            try:
                                for sub in os.listdir(full):
                                    sub_full = os.path.join(full, sub)
                                    if os.path.isdir(sub_full) and any(kw in sub.lower() for kw in keywords):
                                        return sub_full
                            except Exception:
                                pass
                            return full
                        next_paths.append(full)
            except Exception:
                pass
        paths_to_check = next_paths
    return None
def link(src, dst_name, label):
    dst = os.path.join(data_dir, dst_name)
    if os.path.exists(dst):
        print(f'✅ {label}: already linked')
        return
    if src:
        os.symlink(src, dst)
        print(f'✅ {label}: linked from {src}')
    else:
        print(f'❌ {label}: NOT FOUND')

link(find_folder(['ham10000', 'skin-cancer-mnist']),               'ham10000',  'HAM10000')
link(find_folder(['masks', 'segmentation', 'lesion-segmentations']), 'masks', 'HAM10000 Masks')
link(find_folder(['isic-2019', 'isic2019', 'skin-lesion-images', 'andrewmvd', 'salviohexia']), 'isic_2019', 'ISIC 2019')
link(find_folder(['siim-isic', 'melanoma-classification']),        'isic_2020', 'ISIC 2020')
link(find_folder(['isic-2024', 'skin-cancer-detection-3d']),       'isic_2024', 'ISIC 2024')
link(find_folder(['pad-ufes', 'padufes']),                         'pad_ufes',  'PAD-UFES-20')
link(find_folder(['ddi', 'diverse-dermatology']),                  'ddi',       'Stanford DDI')

derm7pt_src = find_folder(['derm7pt', 'release-v0', 'release_v0'])
if derm7pt_src:
    dst = '/kaggle/working/DermaFusion-AI/release_v0'
    if not os.path.exists(dst):
        os.symlink(derm7pt_src, dst)
        print(f'✅ DERM7PT: linked to {dst}')

print('\n📁 data/ contents:', os.listdir(data_dir))

## Step 5: Phase 1 — Segmentation Training (Swin-UNet)
Trains the Swin-Transformer U-Net to perform lesion masking. Saves checkpoints after each epoch to `/kaggle/working/DermaFusion-AI/outputs/weights/resume_seg_checkpoint.pth`. If the session stops, re-running this cell will automatically resume from the last epoch.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
print('=' * 60)
print('🚀 PHASE 1 — Segmentation Training (25 epochs)')
print('   Swin-Tiny UNet | SEG_BATCH_SIZE=8 | 2× T4 GPUs')
print('=' * 60)
!PYTHONPATH=. python train_segmentation.py 2>&1 | tee /kaggle/working/train_segmentation.log
print('\n✅ Phase 1 complete!')

## Step 6: Phase 2 — Classifier Training & Ablations
To run the Ablation Study by training each model from scratch, you can execute the cells below one-by-one. Each cell trains a specific configuration and saves its weights separately so they do not overwrite each other.

### Step 6a: Ablation 1 — No TTA Training (Uses standard training but saves separately)
Estimated training time: ~2 hours per epoch. Trains the full model structure but saves to `best_classifier_no_tta.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation no_tta 2>&1 | tee /kaggle/working/train_no_tta.log
print('\n✅ Ablation 1 training complete!')

### Step 6b: Ablation 2 — ConvNeXt Only Training (Bypasses EVA-02 and fusion)
Estimated training time: ~30 minutes per epoch (much faster as it is a single backbone!). Saves to `best_classifier_convnext_only.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation convnext_only 2>&1 | tee /kaggle/working/train_convnext_only.log
print('\n✅ Ablation 2 training complete!')

### Step 6c: Ablation 3 — EVA-02 Only Training (Bypasses ConvNeXt and fusion)
Estimated training time: ~1.2 hours per epoch. Saves to `best_classifier_eva_only.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation eva_only 2>&1 | tee /kaggle/working/train_eva_only.log
print('\n✅ Ablation 3 training complete!')

### Step 6d: Ablation 4 — No Cross-Attention Training (Fuses via simple average)
Estimated training time: ~2 hours per epoch. Saves to `best_classifier_no_attention.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation no_attention 2>&1 | tee /kaggle/working/train_no_attention.log
print('\n✅ Ablation 4 training complete!')

### Step 6e: Ablation 5 — No Segmentation Training (Original image to ConvNeXt)
Estimated training time: ~2 hours per epoch. Saves to `best_classifier_no_segmentation.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py --ablation no_segmentation 2>&1 | tee /kaggle/working/train_no_segmentation.log
print('\n✅ Ablation 5 training complete!')

### Step 6f: Full Model Training (Dual-Branch + Segmentation + Cross-Attention + TTA)
Estimated training time: ~2 hours per epoch. This is your main final model. Saves to `best_dual_branch_fusion.pth`.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python train_classifier.py 2>&1 | tee /kaggle/working/train_classifier.log
print('\n✅ Full Model training complete!')

## Step 7: Export Weights, Results and Plots to Output Tab
Copies the trained model weights, the **Ablation Study results CSV**, the **Evaluation Dashboard**, and all individual plots to `/kaggle/working/` so they appear in the Kaggle Output panel on the right for easy download.

In [ ]:
import shutil, os

# 1. Export Weights
weights_src = '/kaggle/working/DermaFusion-AI/outputs/weights/'
if os.path.exists(weights_src):
    for f in os.listdir(weights_src):
        if f.endswith('.pth'):
            dst = f'/kaggle/working/{f}'
            shutil.copy(os.path.join(weights_src, f), dst)
            print(f'✅ Weights: {f}  ({os.path.getsize(dst)/1e6:.0f} MB)')
else:
    print('⚠️ No weights folder found (yet)')

# 2. Export Ablation CSV Results
csv_path = '/kaggle/working/DermaFusion-AI/outputs/ablation_study_results.csv'
if os.path.exists(csv_path):
    shutil.copy(csv_path, '/kaggle/working/ablation_study_results.csv')
    print('✅ Results: ablation_study_results.csv')

# 3. Export Plots and Dashboard
plots_src = '/kaggle/working/DermaFusion-AI/outputs/plots/'
if os.path.exists(plots_src):
    for f in os.listdir(plots_src):
        if f.endswith(('.png', '.jpg', '.jpeg')):
            shutil.copy(os.path.join(plots_src, f), f'/kaggle/working/{f}')
            print(f'✅ Plot: {f}')

print('\n📦 Files exported to /kaggle/working/ and ready for download!')

## Step 8: Evaluation & Ablation Study Results
You can evaluate each of your trained ablation configurations individually using the cells below. Make sure you have completed the corresponding training step first so that the weights exist.

### Step 8a: Evaluate Ablation 1 — No TTA (Evaluates the standard weights without TTA)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation no_tta 2>&1 | tee /kaggle/working/evaluate_no_tta.log
print('\n✅ Ablation 1 evaluation complete!')

### Step 8b: Evaluate Ablation 2 — ConvNeXt Only (Bypasses EVA-02)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation convnext_only 2>&1 | tee /kaggle/working/evaluate_convnext_only.log
print('\n✅ Ablation 2 evaluation complete!')

### Step 8c: Evaluate Ablation 3 — EVA-02 Only (Bypasses ConvNeXt)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation eva_only 2>&1 | tee /kaggle/working/evaluate_eva_only.log
print('\n✅ Ablation 3 evaluation complete!')

### Step 8d: Evaluate Ablation 4 — No Cross-Attention (Fuses via simple average)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation no_attention 2>&1 | tee /kaggle/working/evaluate_no_attention.log
print('\n✅ Ablation 4 evaluation complete!')

### Step 8e: Evaluate Ablation 5 — No Segmentation (Original image passed to ConvNeXt)

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py --ablation no_segmentation 2>&1 | tee /kaggle/working/evaluate_no_segmentation.log
print('\n✅ Ablation 5 evaluation complete!')

### Step 8f: Evaluate Full Model (Main Dual-Branch + TTA)
Running this cell evaluates the final model, runs the Ablation Study comparison script, and generates the final dashboard.

In [ ]:
%cd /kaggle/working/DermaFusion-AI
!PYTHONPATH=. python evaluate.py 2>&1 | tee /kaggle/working/evaluate_full.log
print('\n✅ Full Model evaluation and summary dashboard complete!')

## Step 9: Advanced Paper Evaluations
Runs calibration, bootstrap confidence intervals, McNemar significance tests, inference benchmarking, 5-fold CV, and external smartphone/dermoscopy evaluations (PAD-UFES-20 and DERM7PT).

In [ ]:
%cd /kaggle/working/DermaFusion-AI

print('=== 1. Probability Calibration (Temperature Scaling) ===')
!PYTHONPATH=. python -m evaluation.run_temperature_scaling

print('\n=== 2. Statistical Significance (McNemar\'s Test) ===')
!PYTHONPATH=. python -m evaluation.run_statistical_tests

print('\n=== 3. Statistical Confidence Intervals (Bootstrap) ===')
!PYTHONPATH=. python -m evaluation.run_confidence_intervals

print('\n=== 4. Inference Latency & Speed Benchmark ===')
!PYTHONPATH=. python -m evaluation.run_inference_benchmark

print('\n=== 5. 5-Fold Cross-Validation (EVA-02 Small) ===')
!PYTHONPATH=. python -m evaluation.run_cross_validation

print('\n=== 6. Smartphone External Test (PAD-UFES-20) ===')
!PYTHONPATH=. python test_padufes.py

print('\n=== 7. Smartphone Head Fine-Tuning ===')
!PYTHONPATH=. python finetune_padufes.py

print('\n=== 8. DERM7PT External Test (4 combinations) ===')
!PYTHONPATH=. python test_both_weights.py

print('\n✅ All advanced paper evaluations are complete!')